IMPORTING LIBRARIES

In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

from tools import (get_current_time,calculate,TIME_TOOL_SCHEMA,CALCULATOR_TOOL_SCHEMA) # Importing functions and schema from tools.py file.

import numpy as np
import google.generativeai as genai
import json
import os

In [ ]:
import tools

print(dir(tools))

LOAD ENVIRONMENT VARIABLES

In [ ]:
load_dotenv()

Gemini_API_Key = os.getenv("Gemini_API_Key")

if not Gemini_API_Key:
    raise ValueError("Gemini_API_Key Not found in .env file")

print("API Key loaded succesfully!")

CONFIGURING GROQ MODEL

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(GROQ_API_KEY is not None)

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0
)

In [ ]:
response = llm.invoke(
    "What is the capital of France?"
)

print(response.content)

LOADING THE PDF DOCUMENT

In [ ]:
pdf_path = "/home/nineleaps/Documents/Python Files/Agentic_AI/Conversational_RAG_Assistant/sample_rag_dataset.pdf"

reader = PdfReader(pdf_path)

print(f"Total number of pages : {len(reader.pages)}")

EXTRACTING TEXT FROM PDF

In [ ]:
document_text = ""

for page in reader.pages:

    page_text = page.extract_text()

    if page_text:
        document_text+= page_text + "\n"

print(document_text[:500]) # Print the first 500 characters of the extracted text to verify that it has been extracted correctly.

DOCUMENT CHUNKING

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 100)

chunks = text_splitter.split_text(document_text)

print(f"Total chunks Created : {len(chunks)}")

print(chunks[0]) # Reviewing the first chunk to verify that the text has been split correctly.

CREATING EMBEDDING MODEL

In [ ]:
# Selected model for generating embeddings is MiniLM

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generating embeddings for the chunks

embeddings = embedding_model.encode(chunks)

# Converting the embeddings to a numpy array for use with FAISS

embeddings = np.array(embeddings, dtype = np.float32)

print(embeddings.shape)

CREATING FAISS VECTOR INDEX

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

# Adding vectors to the Faiss index

index.add(embeddings)

print(f"Total number of vectors in the index : {index.ntotal}") # Verifying the total number of vectors in the index.

TESTING THE SEMANTIC SEARCH

In [ ]:
query = "What is RAG?" # Sample query to test the retrieval of relevant chunks from the Faiss index.

# Creating query embedding.

query_embeddings = embedding_model.encode([query])

# Converting the query embedding to a numpy array for use with FAISS

query_embedding = np.array(query_embeddings, dtype = np.float32)

# Searching the Faiss index for the top 3 most relevant chunks based on the query embedding.

distances, indices = index.search(query_embeddings, k = 3) 

for idx in indices[0]:
    print("\n")
    print(chunks[idx]) # Printing the retrieved chunks to verify that they are relevant to the query.

CONVERSATION MEMORY

In [ ]:
chat_history = []

# Creating a Helper function which help in formatting the chat history

def format_chat_history(chat_history):

    if not chat_history:
        return "No Previous Conversation."
    
    formatted_history = ""

    for message in chat_history:

        formatted_history += f"{message['role'].capitalize()} : {message['content']}\n"

    return formatted_history

# Testing the format_chat_history function with some sample chat history.

print(format_chat_history(chat_history))

HISTORY AWARE QUESTION RE-WRITING

In [ ]:
def rewrite_question(chat_history, follow_up_question):

    formatted_history = format_chat_history(
        chat_history
    )

    prompt = f"""
You are a query rewriting assistant.

Your task is to convert follow-up questions into standalone questions.

Use conversation history to resolve references such as:

- it
- this
- that
- they
- those
- its

Examples:

User: What is RAG?
User: Explain it more.

Output:
Explain Retrieval Augmented Generation in more detail.

Conversation History:
{formatted_history}

Current Question:
{follow_up_question}

Return ONLY the rewritten standalone question.
"""

    response = llm.invoke(prompt)

    return response.content.strip()

RETRIEVING RELEVANT CHUNKS

In [ ]:
def retrieve_chunks(question, k = 3):

    # Generating embedding for the question

    question_embedding = embedding_model.encode([question])

    question_embedding = np.array(question_embedding, dtype = np.float32)

    # Searching the Faiss index for the top k most relevant chunks based on the question embedding.

    distances, indices = index.search(question_embedding, k = k)

    retrieved_chunks = [chunks[idx] for idx in indices[0]]

    return retrieved_chunks, distances

CREATING CONTEXT FROM RETRIEVED CHUNKS

In [ ]:
def create_context(retrieved_chunks):

    context = "\n\n".join(retrieved_chunks)

    return context

GENERATING ANSWER USING RETRIEVED CONTEXT

In [ ]:
def generate_rag_answer(question, context):

    prompt = f"""

    You are a helpful assistant.

    Answer the question ONLY using the provided context. If the answer is not available in the context,

    say : "I could not find this information in the document."IFrame

    Context : {context}

    Question : {question}
    """
    response = llm.invoke(prompt)
    return response.content

COMPLETE RAG PIPELINE FUNCTION

In [ ]:
def rag_pipeline(follow_up_question):

    standalone_question = rewrite_question(chat_history, follow_up_question)

    retrieved_chunks = retrieve_chunks(standalone_question)

    context = create_context(retrieved_chunks)

    answer = generate_rag_answer(standalone_question, context)

    return answer

UPDATE MEMORY FUNCTION

In [ ]:
def update_chat_history(follow_up_question, assistant_response):

    chat_history.append({"role" : "user", "content" : follow_up_question})

    chat_history.append({"role" : "assistant", "content" : assistant_response})

AVAILABLE TOOLS

In [ ]:
AVAILABLE_TOOLS = [TIME_TOOL_SCHEMA, CALCULATOR_TOOL_SCHEMA]

EXECUTE TOOL FUNCTION

In [ ]:
def execute_tool(tool_name, arguments):

    if tool_name == "get_current_time":
        return get_current_time()
    
    elif tool_name == "calculate":
        expression = arguments.get("expression")

        return calculate(expression)
    else : 
        return{"error" : f"Unknown tool : {tool_name}"}

ROUTING LOGIC

In [ ]:
def should_use_rag(distance, threshold = 1.0):

    best_distance = distance[0][0]

    return best_distance < threshold

TOOL DECISION PROMPT (Gemini will decide whether tool is required or Not)

In [ ]:
def detect_tool_call(
    user_question
):

    prompt = f"""
You are an AI assistant.

Available Tools:

1. get_current_time
   - Returns current date and time

2. calculate
   - Solves mathematical expressions

User Question:
{user_question}

If a tool is needed,
return ONLY JSON.

Examples:

{{

"name":"get_current_time",
"arguments":{{}}
}}

or

{{

"name":"calculate",
"arguments":{{"expression":"25+10"}}
}}

If no tool is needed,
return:

NONE
"""

    response = llm.invoke(
        prompt
    )

    return response.content

PARSE TOOL REQUEST

In [ ]:
import json

def parse_tool_response(response_text):

    if response_text.strip().upper() == "NONE":
        return None

    try:

        cleaned_response = (
            response_text
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )

        return json.loads(cleaned_response)

    except Exception as e:

        print("Tool Parsing Error:", e)

        return None

CONVERTING TOOL RESULT INTO NATURAL LANGUAGE

In [ ]:
def generate_tool_answer(follow_up_question, tool_result):

    prompt = f"""
    You are a helpful assistant. A user has asked the following question : "{follow_up_question}". You have executed a tool to assist in answering this question, and the result from that tool is : "{tool_result}". Using this information, generate a comprehensive answer to the user's question.

    Follow-up Question : {follow_up_question}

    Tool Result : {tool_result}
    """

    response = llm.invoke(prompt)

    return response.content

COMPLETE AGENT PIPELINE

In [ ]:
def agent_pipeline(follow_up_question):

    print("\n" + "="*60)
    print("NEW USER QUESTION")
    print("="*60)

    # Step 1: Rewrite Follow-up Question
    standalone_question = rewrite_question(
        chat_history,
        follow_up_question
    )

    print("\n[STEP 1] STANDALONE QUESTION")
    print(standalone_question)

    # Step 2: Detect Tool Call
    tool_request = detect_tool_call(
        standalone_question
    )

    print("\n[STEP 2] RAW TOOL RESPONSE")
    print(tool_request)

    # Step 3: Parse Tool Response
    parsed_tool = parse_tool_response(
        tool_request
    )

    print("\n[STEP 3] PARSED TOOL")
    print(parsed_tool)

    # -------------------------------------------------
    # TOOL BRANCH
    # -------------------------------------------------

    if parsed_tool:

        print("\n[ROUTING] TOOL SELECTED")

        tool_name = parsed_tool["name"]
        arguments = parsed_tool["arguments"]

        print(f"\nTool Name: {tool_name}")
        print(f"Arguments: {arguments}")

        tool_result = execute_tool(
            tool_name,
            arguments
        )

        print("\n[STEP 4] TOOL RESULT")
        print(tool_result)

        answer = generate_tool_answer(
            follow_up_question,
            tool_result
        )

        source = "TOOL"

    # -------------------------------------------------
    # RAG BRANCH
    # -------------------------------------------------

    else:

        print("\n[ROUTING] NO TOOL DETECTED")
        print("Proceeding with RAG Retrieval...")

        retrieved_chunks, distances = retrieve_chunks(
            standalone_question
        )

        print("\n[STEP 4] RETRIEVAL DISTANCES")
        print(distances)

        print("\n[STEP 5] RETRIEVED CHUNKS")

        for i, chunk in enumerate(retrieved_chunks, start=1):

            print(f"\nChunk {i}:")
            print("-"*40)
            print(chunk[:300])
            print("-"*40)

        use_rag = should_use_rag(
            distances
        )

        print("\n[STEP 6] SHOULD USE RAG?")
        print(use_rag)

        if use_rag:

            print("\n[ROUTING] USING RAG")

            context = create_context(
                retrieved_chunks
            )

            answer = generate_rag_answer(
                standalone_question,
                context
            )

            source = "RAG"

        else:

            print("\n[ROUTING] NO RELEVANT CONTEXT FOUND")

            answer = (
                "I could not find this information in the document."
            )

            source = "NONE"

    # Step 7: Update Memory
    update_chat_history(
        follow_up_question,
        answer
    )

    print("\n[STEP 7] CHAT HISTORY UPDATED")
    print(f"Total Messages Stored: {len(chat_history)}")

    print("\n[FINAL ANSWER]")
    print(answer)

    print("\n" + "="*60)

    return answer, source

RUNNING CONERSATIONAL ASSISTANT (Final Chat Loop)

In [ ]:
tool_response = detect_tool_call(
    "What is 25 * 40?"
)

print(tool_response)

In [ ]:
parsed_tool = parse_tool_response(
    tool_response
)

print(parsed_tool)

In [ ]:
# while True : 

#     follow_up_question = input("User : ")

#     if follow_up_question.lower() in ["exit", "quit"]:
#         print("Exiting the assistant. Goodbye!")
#         break

#     answer, source = agent_pipeline(follow_up_question)

#     print(f"Assistant ({source}) : {answer}")

while True:

    follow_up_question = input("\nUser: ")

    if follow_up_question.lower() in ["exit", "quit"]:
        print("Exiting the assistant. Goodbye!")
        break

    print("\n----- CHAT HISTORY -----")
    print(chat_history)

    answer, source = agent_pipeline(
        follow_up_question
    )

    print(f"\nAssistant ({source}): {answer}")